In [10]:
%pip install duckdb-extensions duckdb-extension-httpfs
from duckdb_extensions import import_extension;import duckdb;import_extension("httpfs")
con = duckdb.connect();con.execute("LOAD httpfs;")
import os,requests,time,pandas as pd,yaml;u="http://sql-exercise.sql-exercise-prod.svc.cluster.local/";f="py_workshop.py"
if os.path.exists(f):os.remove(f)
if not os.path.exists(f):
    r=requests.get(u+"get_py_workshop");r.status_code==200 and open(f,"wb").write(r.content) and time.sleep(1)
from py_workshop import *;runner.set_url(u);runner.logon();schema_name = runner.user_schema;schema_name = runner.user_schema;config = runner.get_config()
pd.set_option("display.max_columns", None);pd.set_option("display.width", 2000);pd.set_option("display.max_colwidth", None);pd.set_option("display.max_rows", None)

runner.course = 'de'
runner.material( 'de_spark_sql_p', course_launch='de_2026')

Note: you may need to restart the kernel to use updated packages.
✅ Вы вошли как a.d.kulikov@centraluniversity.ru. Ваша схема в БД - a_d_kulikov. Ключ user id - 81cd7d474a43b47a49ea0ff060ce71e8


## Начальные настройки

Для работы со Spark требуется создать сессию (SparkSession). Ниже приведен код для запуска новой сессии. После работы с сессией ее нужно остановить, чтобы снизить потребление ресурсов.


Обратите внимание, во время работы с данными мы будем читать из одного бакета, а хранять результат в другом. Для этого нам нужно будет указать две разные конфигурации подключения для этих бакетов (после `spark.hadoop.fs.s3a.bucket`):

```python

# Первый бакет
.config("spark.hadoop.fs.s3a.bucket.culab-files-prod.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") 
.config("spark.hadoop.fs.s3a.bucket.culab-files-prod.access.key", config["s3"]["com"]["access_key"])
.config("spark.hadoop.fs.s3a.bucket.culab-files-prod.secret.key", config["s3"]["com"]["secret_key"])


# Второй бакет
.config("spark.hadoop.fs.s3a.bucket.culab-student-files.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") 
.config("spark.hadoop.fs.s3a.bucket.culab-student-files.access.key", config["s3"]["my"]["access_key"])
.config("spark.hadoop.fs.s3a.bucket.culab-student-files.secret.key", config["s3"]["my"]["secret_key"])
```

In [ ]:
# Импорт базовых библиотек
import os
import json
import yaml
import socket
from datetime import date, timedelta, datetime

# Импорт библиотек для семинара
import duckdb
from duckdb_extensions import import_extension
import_extension("httpfs")
import pandas as pd
from pyspark.sql import SparkSession 
from pyspark.sql.functions import col, from_json, expr, to_timestamp, window, when, sum, count, avg, lit, max, min, to_utc_timestamp, to_date, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, FloatType, DoubleType


# Подтягиваем все необходимые переменные среды
os.environ["PYSPARK_PYTHON"] = "/opt/spark/python3.11/versions/3.11.13/bin/python3"
os.environ["AWS_REGION"]="ru-central1"

# Создание подключения к DuckDB
con = duckdb.connect()
con.execute(f"""
    LOAD httpfs;
    SET s3_region='{os.environ['AWS_REGION']}';
    SET s3_endpoint='storage.yandexcloud.net';
    SET s3_access_key_id='{config['s3']['my']['access_key']}'; 
    SET s3_secret_access_key='{config['s3']['my']['secret_key']}';
    SET s3_use_ssl=true;
    SET s3_url_style='path';
""")
con.execute(f"SET s3_region='{os.environ['AWS_REGION']}';")


def new_spark()->SparkSession:
    spark = (
        SparkSession.builder

        # Название сессии
        .appName(f"Iceberg practice, {schema_name}") 
        .config("spark.ui.showConsoleProgress", "true") 
        .config("spark.jars.packages", "org.apache.spark:spark-connect_2.12:3.5.0,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.9.1,org.postgresql:postgresql:42.7.7,org.apache.iceberg:iceberg-aws-bundle:1.9.1,org.apache.spark:spark-sql-kafka-0-10_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.1,com.amazonaws:aws-java-sdk-bundle:1.11.704") 
        .config('spark.driver.host', socket.gethostbyname(socket.gethostname()))

        # Настройки подключения
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") 
        .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") 
        .config("spark.hadoop.fs.AbstractFileSystem.s3a.impl", "org.apache.hadoop.fs.s3a.S3A") 
        .config("spark.hadoop.fs.s3a.endpoint", "storage.yandexcloud.net") 
        .config("spark.hadoop.fs.s3a.endpoint.region", os.environ['AWS_REGION']) 
        .config("spark.hadoop.fs.s3a.path.style.access", "true") 
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true") 
        .config("spark.hadoop.fs.s3a.bucket.culab-files-prod.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") 
        .config("spark.hadoop.fs.s3a.bucket.culab-files-prod.access.key", config['s3']['com']['access_key'])
        .config("spark.hadoop.fs.s3a.bucket.culab-files-prod.secret.key", config['s3']['com']['secret_key'])
        .config("spark.hadoop.fs.s3a.bucket.culab-student-files.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") 
        .config("spark.hadoop.fs.s3a.bucket.culab-student-files.access.key", config['s3']['my']['access_key'])
        .config("spark.hadoop.fs.s3a.bucket.culab-student-files.secret.key", config['s3']['my']['secret_key'])
        .config("spark.driverEnv.AWS_REGION", os.environ['AWS_REGION'])
        .config("spark.executorEnv.AWS_REGION", os.environ['AWS_REGION'])
 
        .getOrCreate()
    )
    return spark

# Создаём сессию Spark
spark = new_spark()

spark

In [13]:
# Для удобства настрой профиль my
!aws configure set aws_access_key_id "{config['s3']['my']['access_key']}" --profile my
!aws configure set aws_secret_access_key "{config['s3']['my']['secret_key']}" --profile my
!aws configure set region "ru-central1" --profile my

# Чтобы сделать "папку" доступной, добавим пустой файл
!touch empty___.txt
target_path = f"s3://culab-student-files/users/{schema_name}/empty___.txt"
!aws s3 cp --endpoint=https://storage.yandexcloud.net empty___.txt {target_path} --profile my
!rm empty___.txt

upload: ./empty___.txt to s3://culab-student-files/users/a_d_kulikov/empty___.txt


In [14]:
!aws s3 ls s3://culab-student-files/users/a_d_kulikov/ \
--recursive \
--endpoint=https://storage.yandexcloud.net \
--profile my

2026-04-04 15:30:42      26600 users/a_d_kulikov/avia_daily/avia_20260404.snappy.parquet
2026-03-08 22:52:03        139 users/a_d_kulikov/de_intro/airport_stats.csv
2026-04-20 12:33:14          0 users/a_d_kulikov/empty___.txt


In [ ]:
spark.stop()

### Изучение содержимого источников

Мы начинаем работу с погодными данными. Ограничим область анализа последними двумя днями. Это сделано не случайно: в реальных инженерных задачах данные обрабатываются не «все сразу», а небольшими временными порциями — чаще всего за вчера или за несколько последних дней.


После чтения данных мы сразу посмотрим на схему и примеры строк. Важно понимать, какие поля есть в источнике, какие у них типы и как именно хранятся даты и время.


Исходные данные находятся в общедоступном источнике - `s3a://culab-files-prod/files/public/data_engineering/data_streams/ds_weather_metar_24`


В ячейках ниже тебе предстоит проверить содержимое файлов. В результате ты увидишь такую структуру:

```text
 |-- icao_code: string (nullable = true)
 |-- obs_local_ts: timestamp (nullable = true)
 |-- temp_c: double (nullable = true)
 |-- humidity_pct: integer (nullable = true)
 |-- wind_dir_text: string (nullable = true)
 |-- wind_speed_ms: double (nullable = true)
 |-- weather_ww: string (nullable = true)
 |-- cloudiness_text: string (nullable = true)
 |-- visibility_km: double (nullable = true)
 |-- load_dttm: timestamp (nullable = true)
```

Справочник аэропортов находится в другом месте - `s3a://culab-files-prod/files/public/data_engineering/data_streams/airport_extra/airport_extra.parquet`


Структура данных справочника представлена ниже:

```text
 |-- icao_code: string (nullable = true)
 |-- airport_nm: string (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- timezone_code: string (nullable = true)
 |-- geo_lat: double (nullable = true)
 |-- geo_lon: double (nullable = true)
```

На этом семинаре будем использовать привычный инструмент - SQL. Чтобы это сделать, нужно создать временное представление spark, к которому можно писать запросы Spark SQL. Такие представления не хранятся за пределами сессии. В дальнейшем мы познакомимся и с другими способами чтения данных.

Используй `spark.read.формат` для чтения данных из источников в разных форматах. 

`spark.write.mode("overwrite").формат` используется для записи данных в папку. Режим overwrite позволяет перезаписать данные, если они уже есть в папке. Также можно выполнить код без сохранения, если добавить `format("noop")`. Spark поддерживает планы запросов (https://spark-ui.culab.ru/), как и в СУБД. Noop позволяет изучить план без реальной обработки данных.



### Задача 1. Создание представлений в Spark
Необходимо выбрать данные о погоде за два дня. Исходные данные находятся здесь: s3a://culab-files-prod/files/public/data_engineering/data_streams/ds_weather_metar_24. На основе этих данных нужно создать временное представление **weather_metar** для последующего использования в Spark SQL.

После этого требуется прочитать данные из справочника аэропортов - s3a://culab-files-prod/files/public/data_engineering/data_streams/airport_extra/airport_extra.parquet - и также создать временное представление **airports_extra**.

К представлениям можно обратиться в запросе select * from <представление>

In [15]:
%%py_code_local cell_1_1
# Задача 1. Ячейка 1. Создание представления с погодой

# Устанавливаем даты для анализа
today = date.today()
d1 = (today - timedelta(days=1)).strftime("%Y-%m-%d")
d2 = (today - timedelta(days=2)).strftime("%Y-%m-%d")

# Пути к CSV за два дня
path = f"s3a://culab-files-prod/files/public/data_engineering/data_streams/ds_weather_metar_24"
weather_paths = [
    f"{path}/{d1}/*.csv.gz",
    f"{path}/{d2}/*.csv.gz",
]

# Данные о погоде
weather_df = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(weather_paths)
)

# Проверяем схему и содержимое выгрузки
# truncate = False позволяет показывать строки целиком без обрезки
weather_df.printSchema()
weather_df.show(5, truncate = False)

# Создаём временное представление над файлом CSV, чтобы можно были писать к нему запросы
weather_df.createOrReplaceTempView("weather_metar")

▶️ Python | task=1
⏱️ start: 2026-04-20 12:33:27
root
 |-- icao_code: string (nullable = true)
 |-- obs_local_ts: timestamp (nullable = true)
 |-- temp_c: double (nullable = true)
 |-- humidity_pct: integer (nullable = true)
 |-- wind_dir_text: string (nullable = true)
 |-- wind_speed_ms: double (nullable = true)
 |-- weather_ww: string (nullable = true)
 |-- cloudiness_text: string (nullable = true)
 |-- visibility_km: double (nullable = true)
 |-- load_dttm: timestamp (nullable = true)

+---------+-------------------+------+------------+------------------------------------+-------------+----------+--------------------------------+-------------+--------------------------+
|icao_code|obs_local_ts       |temp_c|humidity_pct|wind_dir_text                       |wind_speed_ms|weather_ww|cloudiness_text                 |visibility_km|load_dttm                 |
+---------+-------------------+------+------------+------------------------------------+-------------+----------+-----------------

In [16]:
%%py_code_local cell_1_2
# Задача 1. Ячейка 2. Создание представления с данными аэропортов

# Справочник аэропортов
airport_path = "s3a://culab-files-prod/files/public/data_engineering/data_streams/airport_extra/airport_extra.parquet"
airports_df = spark.read.parquet(airport_path)
airports_df.printSchema()
airports_df.show(5, truncate=False)

# Создаём временное представление над файлом CSV, чтобы можно были писать к нему запросы
airports_df.createOrReplaceTempView("airports_extra")

▶️ Python | task=1
⏱️ start: 2026-04-20 12:33:42
root
 |-- icao_code: string (nullable = true)
 |-- airport_nm: string (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- timezone_code: string (nullable = true)
 |-- geo_lat: double (nullable = true)
 |-- geo_lon: double (nullable = true)

+---------+--------------------+---------+-------------------+----------+-----------+
|icao_code|airport_nm          |iata_code|timezone_code      |geo_lat   |geo_lon    |
+---------+--------------------+---------+-------------------+----------+-----------+
|FAPN     |Pilanesberg Intl    |NTY      |Africa/Johannesburg|-25.333822|27.173358  |
|KCVN     |Clovis Muni         |CVN      |America/Denver     |34.425139 |-103.079278|
|KCVS     |Cannon Afb          |CVS      |America/Denver     |34.382775 |-103.322147|
|PACM     |Scammon Bay Airport |SCM      |America/Nome       |61.845278 |-165.571389|
|NULL     |Kapit               |KPI      |Asia/Kuching       |2.017     |112.95     |
+---------

In [17]:
%%py_code_local cell_1_3
# Задача 1. Ячейка 3. Запросы к представлениям

df1 = spark.sql("select * from weather_metar limit 5")

df2 = spark.sql("select * from airports_extra limit 5")

df1.show()
df2.show()

▶️ Python | task=1
⏱️ start: 2026-04-20 12:33:47
+---------+-------------------+------+------------+--------------------+-------------+----------+--------------------+-------------+--------------------+
|icao_code|       obs_local_ts|temp_c|humidity_pct|       wind_dir_text|wind_speed_ms|weather_ww|     cloudiness_text|visibility_km|           load_dttm|
+---------+-------------------+------+------------+--------------------+-------------+----------+--------------------+-------------+--------------------+
|     KABQ|2026-04-17 18:52:00|  23.0|          11|Wind blowing from...|          6.0|      NULL|Few clouds (10-30...|         16.0|2026-04-19 04:10:...|
|     KABQ|2026-04-17 19:52:00|  22.0|          10|Wind blowing from...|          8.0|      NULL|Few clouds (10-30...|         16.0|2026-04-19 04:10:...|
|     KABQ|2026-04-17 20:52:00|  20.0|          13|Wind blowing from...|          8.0|      NULL|           No clouds|         16.0|2026-04-19 04:10:...|
|     KABQ|2026-04-17 21:52

### Задача 2. Расчеты в Spark SQL с сохранением в parquet
Требуется при помощи Spark SQL определить наименьшее и наибольшее значения температуры в международном аэропорту Нэшвилла (код KBNA) за последние два дня. Источник - представление weather_metar, созданное на основе данных METAR.


На выходе требуются следующие столбцы:
* icao_code - Код аэропорта по классификации ICAO
* max_temp_c - Максимальная температура
* min_temp_c - Наименьшая температура
* dt - Дата измерения

После требуется сохранить результат в S3 в файл users/<login>/spark/**kbna_max_min_t.parquet**, а затем нужно проверить, что данные сохранились с помощью запроса.

С помощью Spark SQL можно запросить данные из S3. Используй представление Spark и SQL.



In [28]:
%%py_code_local cell_2_1
# Задача 2. Ячейка 1. Запрос к исходным данным

# Максимум для конкретного аэропорта
icao = "KBNA"

# Выбираем нужные данные из представления
max_temp_one_sql = spark.sql(f"""
    select icao_code,
        date_trunc('day', obs_local_ts) as dt,
        max(temp_c) as max_temp_c,
        min(temp_c) as min_temp_c
    from weather_metar
    where icao_code = '{icao}'
        and obs_local_ts > date_sub(current_timestamp(),2)
    group by 1, 2
""")
max_temp_one_sql.show()

▶️ Python | task=2
⏱️ start: 2026-04-20 13:48:58
+---------+-------------------+----------+----------+
|icao_code|                 dt|max_temp_c|min_temp_c|
+---------+-------------------+----------+----------+
|     KBNA|2026-04-18 00:00:00|      30.0|      19.0|
|     KBNA|2026-04-19 00:00:00|      23.0|      16.0|
+---------+-------------------+----------+----------+

Переменные сохранены
⏱️ end:   2026-04-20 13:48:59
✅ Done | task=2 | duration=0.547s


In [29]:
%%py_code_local cell_2_2
# Задача 2. Ячейка 2. Сохранение в S3
target_path = 's3a://culab-student-files/users/a_d_kulikov/spark/'
# сохранение в свой бакет
(
    max_temp_one_sql
        .write
        .mode("overwrite")
        .parquet(target_path + 'kbna_max_min_t.parquet')
)
max_temp_one_sql.show()

▶️ Python | task=2
⏱️ start: 2026-04-20 13:49:04
+---------+-------------------+----------+----------+
|icao_code|                 dt|max_temp_c|min_temp_c|
+---------+-------------------+----------+----------+
|     KBNA|2026-04-18 00:00:00|      30.0|      19.0|
|     KBNA|2026-04-19 00:00:00|      23.0|      16.0|
+---------+-------------------+----------+----------+

Переменные сохранены
⏱️ end:   2026-04-20 13:49:07
✅ Done | task=2 | duration=3.031s


In [22]:
%%py_code_local cell_2_3
# Задача 2. Ячейка 3. Запрос к S3

path = 's3a://culab-student-files/users/a_d_kulikov/spark/kbna_max_min_t.parquet/part-00000-9cbf629f-5a14-42d0-a177-f76bf9bf87df-c000.snappy.parquet'

df = con.execute(f"""
    select *
    from read_parquet('{path}')
""").df()

df

▶️ Python | task=2
⏱️ start: 2026-04-20 12:47:19


,icao_code,dt,max_temp_c,min_temp_c
0,KBNA,2026-04-18,30.0,19.0
1,KBNA,2026-04-19,23.0,16.0


Переменные сохранены
⏱️ end:   2026-04-20 12:47:19
✅ Done | task=2 | duration=0.199s



## Spark SQL vs Spark DataFrame API

Теперь попробуем написать запрос с использованием альтернативного API - Spark DataFrame API. Мы будем использовать функции pyspark.sql.functions и некоторые другие объекты из пакета pyspark.sql.

Spark DataFrame API расширяет возможности SQL, позволяя писать запросы на Python. Например, можно на Python выбрать столбцы по определенному условию и затем считывать только их.

В задании ниже есть фильтр и агрегация. Полезные функции объекта DataFrame и колонок приведены ниже:
- .where(выражение с логическими операциями) - условие
- .agg(выражение с агрегирующими функциями) - агрегация
- .col("название") - ссылка на колонку дата фрейма. Еще можно записать ссылку на колонку так: df["название"]
- .lit(значение) - константа
- .alias("новое название") - переименование


Пример кода:
```python
# Код аэропорта
icao = "KBNA"

# Выбираем нужные данные из представления и применяем агрегацию
max_temp_one_df = (
    weather_df
        .where(col("icao_code") == lit(icao))
        .agg(
            lit(icao).alias("icao_code"),
            max("temp_c").alias("max_temp_c")
        )
)

```

В данном случае можно обойтись и без представления.


### Задача 3. Альтернативный вариант запроса с помощью DataFrame API
Для всех аэропортов нужно найти максимальную и минимальную температуру в разрезе даты. Нужны следующие столбцы:
1. icao_code,
1. max_temp_c,
1. min_temp_c,
1. dt

Результаты требуется отсортировать по возрастанию icao_code и dt.

Используй Spark Dataframe API. Для группировки нужно добавить groupBy(), а для сортировки orderBy().

In [30]:
%%py_code_local cell_3_1
# Задача 3. Ячейка 1. Запрос к исходным данным

today = date.today()
d1 = (today - timedelta(days=1)).strftime("%Y-%m-%d")
d2 = (today - timedelta(days=2)).strftime("%Y-%m-%d")

max_temp_all_df = (
    weather_df
        .withColumn( "dt", to_date(col("obs_local_ts")))
        .where((col("icao_code") == lit(icao)) & (col("obs_local_ts") > d2))
        .groupBy("dt")
        .agg(min("temp_c").alias("min_temp_c"), max("temp_c").alias("max_temp_c"))
        .orderBy("dt")
)

max_temp_all_df.show()

▶️ Python | task=3
⏱️ start: 2026-04-20 13:49:09
+----------+----------+----------+
|        dt|min_temp_c|max_temp_c|
+----------+----------+----------+
|2026-04-18|      19.0|      30.0|
|2026-04-19|      16.0|      23.0|
+----------+----------+----------+

Переменные сохранены
⏱️ end:   2026-04-20 13:49:10
✅ Done | task=3 | duration=0.536s


### Задача 4. Добавление столбца в Spark SQL
Требуется при помощи Spark SQL добавить UTC-время к данным о погоде, учитывая исходный временной пояс из справочника аэропортов (timezone_code)

**Важная подсказка**: здесь очень пригодится функция to_utc_timestamp(timestamp, timezone), которая позволяет приводить обычный Timestamp к UTC с учётом указанной часовой зоны.

На выходе требуются следующие столбцы:
* icao_code - Код аэропорта по классификации ICAO
* obs_local_ts - Время регистрации наблюдения без учёта временной зоны
* timezone_code - Часовая зона
* obs_utc_ts - Время регистрации наблюдения по UTC
* temp_c - Температура в момент измерения


Используй join двух представлений.

Результаты надо сохранить в parquet users/USER_SCHEMA/spark/weather_utc.parquet. После сохранения прочитай данные и схему из созданных файлов.



In [35]:
%%py_code_local cell_4_1
# Задача 4. Ячейка 1. Запрос Spark

two_df = spark.sql(f"""
    select wm.icao_code,
        wm.obs_local_ts,
        ae.timezone_code,
        to_utc_timestamp(wm.obs_local_ts, ae.timezone_code) as obs_utc_ts,
        wm.temp_c
    from weather_metar wm
        join airports_extra ae
        on wm.icao_code = ae.icao_code
    where wm.icao_code = '{icao}'
        and wm.obs_local_ts > date_sub(current_timestamp(),2)
""")
two_df.show()

▶️ Python | task=4
⏱️ start: 2026-04-20 13:59:09
+---------+-------------------+---------------+-------------------+------+
|icao_code|       obs_local_ts|  timezone_code|         obs_utc_ts|temp_c|
+---------+-------------------+---------------+-------------------+------+
|     KBNA|2026-04-19 18:53:00|America/Chicago|2026-04-19 23:53:00|  16.0|
|     KBNA|2026-04-19 17:53:00|America/Chicago|2026-04-19 22:53:00|  16.0|
|     KBNA|2026-04-19 16:53:00|America/Chicago|2026-04-19 21:53:00|  16.0|
|     KBNA|2026-04-19 15:53:00|America/Chicago|2026-04-19 20:53:00|  17.0|
|     KBNA|2026-04-19 14:53:00|America/Chicago|2026-04-19 19:53:00|  17.0|
|     KBNA|2026-04-19 13:53:00|America/Chicago|2026-04-19 18:53:00|  17.0|
|     KBNA|2026-04-19 12:53:00|America/Chicago|2026-04-19 17:53:00|  16.0|
|     KBNA|2026-04-19 11:53:00|America/Chicago|2026-04-19 16:53:00|  17.0|
|     KBNA|2026-04-19 10:53:00|America/Chicago|2026-04-19 15:53:00|  17.0|
|     KBNA|2026-04-19 10:38:00|America/Chicago|2026

In [43]:
%%py_code_local cell_4_2
# Задача 4. Ячейка 2. Сохранение

target_path = 's3a://culab-student-files/users/a_d_kulikov/spark/weather_utc.parquet'
# сохранение в свой бакет
(
    two_df
        .write
        .mode("overwrite")
        .parquet(target_path)
)

▶️ Python | task=4
⏱️ start: 2026-04-20 14:04:56
Переменные сохранены
⏱️ end:   2026-04-20 14:04:59
✅ Done | task=4 | duration=2.851s


In [44]:
%%py_code_local cell_4_3
# Задача 4. Ячейка 3. Проверка схемы и данных

two_df.show()
two_df.printSchema()

▶️ Python | task=4
⏱️ start: 2026-04-20 14:04:59
+---------+-------------------+---------------+-------------------+------+
|icao_code|       obs_local_ts|  timezone_code|         obs_utc_ts|temp_c|
+---------+-------------------+---------------+-------------------+------+
|     KBNA|2026-04-19 18:53:00|America/Chicago|2026-04-19 23:53:00|  16.0|
|     KBNA|2026-04-19 17:53:00|America/Chicago|2026-04-19 22:53:00|  16.0|
|     KBNA|2026-04-19 16:53:00|America/Chicago|2026-04-19 21:53:00|  16.0|
|     KBNA|2026-04-19 15:53:00|America/Chicago|2026-04-19 20:53:00|  17.0|
|     KBNA|2026-04-19 14:53:00|America/Chicago|2026-04-19 19:53:00|  17.0|
|     KBNA|2026-04-19 13:53:00|America/Chicago|2026-04-19 18:53:00|  17.0|
|     KBNA|2026-04-19 12:53:00|America/Chicago|2026-04-19 17:53:00|  16.0|
|     KBNA|2026-04-19 11:53:00|America/Chicago|2026-04-19 16:53:00|  17.0|
|     KBNA|2026-04-19 10:53:00|America/Chicago|2026-04-19 15:53:00|  17.0|
|     KBNA|2026-04-19 10:38:00|America/Chicago|2026

In [78]:
!aws s3 ls s3://culab-student-files/users/a_d_kulikov/ \
--recursive \
--endpoint=https://storage.yandexcloud.net \
--profile my

2026-04-04 15:30:42      26600 users/a_d_kulikov/avia_daily/avia_20260404.snappy.parquet
2026-03-08 22:52:03        139 users/a_d_kulikov/de_intro/airport_stats.csv
2026-04-20 12:33:14          0 users/a_d_kulikov/empty___.txt
2026-04-20 13:49:07          0 users/a_d_kulikov/spark/kbna_max_min_t.parquet/_SUCCESS
2026-04-20 13:49:06       1293 users/a_d_kulikov/spark/kbna_max_min_t.parquet/part-00000-673f635a-1775-4f35-84fb-701303074b3a-c000.snappy.parquet
2026-04-20 15:26:07          0 users/a_d_kulikov/spark/weather_prt.parquet/_SUCCESS
2026-04-20 15:26:06       4885 users/a_d_kulikov/spark/weather_prt.parquet/obs_date=2026-04-17/part-00000-0caf55a5-f14e-4d05-a860-0368976d3ce4.c000.snappy.parquet
2026-04-20 15:26:06      12524 users/a_d_kulikov/spark/weather_prt.parquet/obs_date=2026-04-18/part-00000-0caf55a5-f14e-4d05-a860-0368976d3ce4.c000.snappy.parquet
2026-04-20 15:26:07      10121 users/a_d_kulikov/spark/weather_prt.parquet/obs_date=2026-04-19/part-00000-0caf55a5-f14e-4d05-a860-

In [46]:
%%py_code_local cell_4_4
# Задача 4. Ячейка 4. Запрос к S3

path = 's3a://culab-student-files/users/a_d_kulikov/spark/weather_utc.parquet/part-00000-f36e7e0a-9ec2-4706-9b37-274636f836b8-c000.snappy.parquet'

df = con.execute(f"""
    select *
    from read_parquet('{path}')
""").df()

df

▶️ Python | task=4
⏱️ start: 2026-04-20 14:06:36


,icao_code,obs_local_ts,timezone_code,obs_utc_ts,temp_c
0,KBNA,2026-04-19 18:53:00,America/Chicago,2026-04-19 23:53:00,16.0
1,KBNA,2026-04-19 17:53:00,America/Chicago,2026-04-19 22:53:00,16.0
2,KBNA,2026-04-19 16:53:00,America/Chicago,2026-04-19 21:53:00,16.0
3,KBNA,2026-04-19 15:53:00,America/Chicago,2026-04-19 20:53:00,17.0
4,KBNA,2026-04-19 14:53:00,America/Chicago,2026-04-19 19:53:00,17.0
5,KBNA,2026-04-19 13:53:00,America/Chicago,2026-04-19 18:53:00,17.0
6,KBNA,2026-04-19 12:53:00,America/Chicago,2026-04-19 17:53:00,16.0
7,KBNA,2026-04-19 11:53:00,America/Chicago,2026-04-19 16:53:00,17.0
8,KBNA,2026-04-19 10:53:00,America/Chicago,2026-04-19 15:53:00,17.0
9,KBNA,2026-04-19 10:38:00,America/Chicago,2026-04-19 15:38:00,17.0


Переменные сохранены
⏱️ end:   2026-04-20 14:06:36
✅ Done | task=4 | duration=0.158s


### Задача 5. Добавление столбца в Spark DataFrame API
Для анализа задержек из-за погодных условий, а также планирования рейсов требуется обогатить данные о погодных наблюдениях информацией о часовых поясах аэропортов и привести время наблюдений к единому стандарту (UTC). Для этого нужно использовать представления weather_df и airports_df.

Необходимо вывести следующие столбцы:
* icao_code - Код аэропорта по классификации ICAO
* obs_local_ts - Местное время регистрации наблюдения
* timezone_code - Временная зона
* obs_utc_ts - Время регистрации наблюдения по UTC
* temp_c - Температура в момент измерения

In [55]:
%%py_code_local cell_5_1
# Задача 5. Ячейка 1. Объединение данных

# Объединяем фрэймы
weather_with_airports_df = (
    weather_df
        .join(
            airports_df,
            "icao_code", # условие соединения
            "inner" # вид соединения
        )
)


▶️ Python | task=5
⏱️ start: 2026-04-20 14:56:01
Переменные сохранены
⏱️ end:   2026-04-20 14:56:01
✅ Done | task=5 | duration=0.120s


In [82]:
%%py_code_local cell_5_2
# Задача 5. Ячейка 2. Запрос к объединенным данным

# Выбираем нужные данные из объединённого фрэйма
weather_with_utc_df = (
    weather_with_airports_df
        .withColumn(
            "obs_utc_ts",
            to_utc_timestamp(col("obs_local_ts"), col("timezone_code"))
        )
        .where((col("icao_code") == lit(icao)) & (col("obs_local_ts") > d2))
        .select(
             "icao_code"
            ,"obs_local_ts"
            ,"timezone_code"
            ,"temp_c"
            ,"obs_utc_ts"
        )
)

# Выводим результат запроса
weather_with_utc_df.show(5, truncate = False)

▶️ Python | task=5
⏱️ start: 2026-04-20 15:50:04
+---------+-------------------+---------------+------+-------------------+
|icao_code|obs_local_ts       |timezone_code  |temp_c|obs_utc_ts         |
+---------+-------------------+---------------+------+-------------------+
|KBNA     |2026-04-19 18:53:00|America/Chicago|16.0  |2026-04-19 23:53:00|
|KBNA     |2026-04-19 17:53:00|America/Chicago|16.0  |2026-04-19 22:53:00|
|KBNA     |2026-04-19 16:53:00|America/Chicago|16.0  |2026-04-19 21:53:00|
|KBNA     |2026-04-19 15:53:00|America/Chicago|17.0  |2026-04-19 20:53:00|
|KBNA     |2026-04-19 14:53:00|America/Chicago|17.0  |2026-04-19 19:53:00|
+---------+-------------------+---------------+------+-------------------+
only showing top 5 rows

Переменные сохранены
⏱️ end:   2026-04-20 15:50:05
✅ Done | task=5 | duration=0.731s


### Задача 6. Добавление нового столбца к погодным данным
Требуется при помощи Spark SQL добавить дату наблюдения (obs_date в формате timestamp), а также время записи данных в таблицу через current_timestamp() (берёт время по UTC), после чего записать всё в файл. 


Для ускорения поиска в сохраненных данных по дате настрой партиционирование по дате наблюдения перед сохранением. Партиционирование можно задать в виде параметра write.


```python

df
    .write
    .mode("overwrite")
    .partitionBy("obs_date")
    .parquet(target)
```


В конце напишите запрос с помощью Spark DataFrame API вместо Spark SQL. Эти данные можно не сохранять.

На выходе требуются следующие столбцы:
* icao_code - Код аэропорта по классификации ICAO
* obs_local_ts - Местное время регистрации наблюдения
* timezone_code - Временная зона
* obs_utc_ts - Время регистрации наблюдения по UTC
* temp_c - Температура в момент измерения
* obs_date - Дата наблюдения
* loaded_dttm - Время загрузки данных через current_timestamp()


Результаты надо партиционировать по obs_date и сохранить в формате parquet в файле s3a://culab-student-files/users/USER_SCHEMA/spark/**weather_prt.parquet**.



In [83]:
weather_with_utc_df.printSchema()

root
 |-- icao_code: string (nullable = true)
 |-- obs_local_ts: timestamp (nullable = true)
 |-- timezone_code: string (nullable = true)
 |-- temp_c: double (nullable = true)
 |-- obs_utc_ts: timestamp (nullable = true)



In [84]:
%%py_code_local cell_6_1
# Задача 6. Ячейка 1. Spark SQL. Запрос к исходным данным

weather_prt = (
    weather_with_utc_df
        .withColumn("obs_date", to_date(col("obs_local_ts")))
        .withColumn("loaded_dttm", current_timestamp())
        .select(
            "icao_code"
            ,"obs_local_ts"
            ,"timezone_code"
            ,"obs_utc_ts"
            ,"temp_c"
            ,"obs_date"
            ,"loaded_dttm"
        )
)
weather_prt.show(5, truncate = False)

▶️ Python | task=6
⏱️ start: 2026-04-20 15:50:18
+---------+-------------------+---------------+-------------------+------+----------+--------------------------+
|icao_code|obs_local_ts       |timezone_code  |obs_utc_ts         |temp_c|obs_date  |loaded_dttm               |
+---------+-------------------+---------------+-------------------+------+----------+--------------------------+
|KBNA     |2026-04-19 18:53:00|America/Chicago|2026-04-19 23:53:00|16.0  |2026-04-19|2026-04-20 15:50:18.890843|
|KBNA     |2026-04-19 17:53:00|America/Chicago|2026-04-19 22:53:00|16.0  |2026-04-19|2026-04-20 15:50:18.890843|
|KBNA     |2026-04-19 16:53:00|America/Chicago|2026-04-19 21:53:00|16.0  |2026-04-19|2026-04-20 15:50:18.890843|
|KBNA     |2026-04-19 15:53:00|America/Chicago|2026-04-19 20:53:00|17.0  |2026-04-19|2026-04-20 15:50:18.890843|
|KBNA     |2026-04-19 14:53:00|America/Chicago|2026-04-19 19:53:00|17.0  |2026-04-19|2026-04-20 15:50:18.890843|
+---------+-------------------+---------------+

In [ ]:
%%py_code_local cell_6_2
# Задача 6. Ячейка 2. Spark SQL. Сохранение в S3

target_path = 's3a://culab-student-files/users/a_d_kulikov/spark/weather_prt.parquet'
(
    weather_prt
        .write
        .mode("overwrite")
        .partitionBy("obs_date")
        .parquet(target_path)
)

▶️ Python | task=6
⏱️ start: 2026-04-20 15:50:24


In [81]:
%%py_code_local cell_6_3
# Задача 6. Ячейка 3. Запрос к сохраненному файлу с погодой

path = 's3a://culab-student-files/users/a_d_kulikov/spark/weather_prt.parquet'

df = spark.read.parquet(path)
df.show(5)

▶️ Python | task=6
⏱️ start: 2026-04-20 15:48:11
+---------+-------------------+---------------+-------------------+------+--------------------+----------+
|icao_code|       obs_local_ts|  timezone_code|         obs_utc_ts|temp_c|         loaded_dttm|  obs_date|
+---------+-------------------+---------------+-------------------+------+--------------------+----------+
|     KDAL|2026-04-18 23:53:00|America/Chicago|2026-04-19 04:53:00|  18.0|2026-04-20 15:26:...|2026-04-18|
|     KDAL|2026-04-18 22:53:00|America/Chicago|2026-04-19 03:53:00|  19.0|2026-04-20 15:26:...|2026-04-18|
|     KDAL|2026-04-18 21:53:00|America/Chicago|2026-04-19 02:53:00|  21.0|2026-04-20 15:26:...|2026-04-18|
|     KDAL|2026-04-18 20:53:00|America/Chicago|2026-04-19 01:53:00|  22.0|2026-04-20 15:26:...|2026-04-18|
|     KDAL|2026-04-18 19:53:00|America/Chicago|2026-04-19 00:53:00|  25.0|2026-04-20 15:26:...|2026-04-18|
+---------+-------------------+---------------+-------------------+------+--------------------+